# GLD ML Prediction Horizon Scan — 실험 보고서
**날짜:** 2026-05-10 ~ 2026-05-11  
**연구자:** Joon Jeremy Chun  
**위치:** `research/horizon_scan_gld/`  

---

## 1. 연구 배경 및 목적

### 전체 시스템 구조 (Layer 0~3)

| Layer | 역할 | 상태 |
|-------|------|------|
| **0** | 인간 매크로 판단 — 어떤 자산을 담을지 | 운영 중 |
| **1** | 전략 최적화 — 4개 전략 패밀리 파라미터 그리드 서치 | 운영 중 |
| **2** | ML 예측 — OLS/Ridge/Lasso/ElasticNet으로 N일 미래 수익 예측 → 포지션 비중 | 운영 중 (130일 고정) |
| **3** | SINDy 비선형 조합 — 수익 최대화 | 미구현 (미래 과제) |

### 현재 시스템의 고정값
현재 Layer 2에서 **130일 미래 수익**을 예측 타겟으로 고정해서 사용 중.  
**핵심 질문:** 130일이 최적인가? 아니면 다른 N일이 더 좋은가?

---

## 2. 실험 설계

### 아이디어: Dense Window + Horizon Scan

**기존 시스템:**
```
훈련 윈도우: 1m, 3m, 6m, 1y (성긴 간격)
예측 타겟: 130일 고정
```

**이번 실험:**
```
훈련 윈도우: 20d ~ 520d (5일 간격, 101개) ← Strategy 1 전용 새 계산
예측 타겟: 5d ~ 500d (5일 간격, 100개) ← 전부 스캔
→ 어떤 N일 타겟이 최적인지 데이터로 확인
```

### Feature 구성 (총 ~1,141개)

| 전략 | Feature 수 | 데이터 출처 |
|------|-----------|------------|
| **S1 Adaptive Band (dense)** | 101 windows × topN | 새로 계산 |
| **S2 MA Crossover** | 기존 horizons × topN | 기존 앵커 재사용 |
| **S3 Adaptive Vol Band** | 기존 horizons × topN | 기존 앵커 재사용 |

각 feature = 해당 파라미터를 selection period에 적용한 **매일의 포지션 신호(0/1)**

### ML 모델 (4가지)
- OLS (선형 회귀, 정규화 없음)
- Ridge (L2 정규화, alpha 30개 CV)
- Lasso (L1 정규화, alpha 30개 CV, 자동 feature 선택)
- ElasticNet (L1+L2 혼합, alpha 30개 × l1_ratio 5개 CV)

**선택 기준:** CV-MSE 최소 모델 선택 (TimeSeriesSplit 5-fold)

### 실험 규모
- 앵커: 10개 (2021-05 ~ 2025-11, 6개월마다)
- Horizon 스캔: 100개 (5~500일)
- top10 실험: 10 × 100 = 1,000 (72분 소요)
- top20 실험: 10 × 100 = 1,000 (82분 소요)
- **자산: GLD만** (다른 자산은 이 결과를 그대로 적용할 예정)

### 독립 실험 원칙
- 폴더: `research/horizon_scan_gld/` — 원본 파이프라인과 완전 분리
- 원본 코드 건드리지 않음
- 데이터 참조만 원본에서 (read-only)

---

## 3. 결과 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# 결과 로드
BASE = '../../research/horizon_scan_gld/outputs/'
t10 = pd.read_csv(BASE + 'horizon_scan_top10_summary.csv').sort_values('horizon_days')
t20 = pd.read_csv(BASE + 'horizon_scan_top20_summary.csv').sort_values('horizon_days')

print(f'top10: {len(t10)}개 horizon')
print(f'top20: {len(t20)}개 horizon')
print()
print('=== top10 Top5 (Forward Return) ===')
print(t10.nlargest(5,'avg_forward_return')[['horizon_days','avg_forward_return','avg_cv_mse','dir_accuracy','best_model_mode']].to_string(index=False))
print()
print('=== top20 Top5 (Forward Return) ===')
print(t20.nlargest(5,'avg_forward_return')[['horizon_days','avg_forward_return','avg_cv_mse','dir_accuracy','best_model_mode']].to_string(index=False))

In [ ]:
# ============================================================
# Graph 1 & 2: top10
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Top-10 Features  |  4 Models (OLS/Ridge/Lasso/ElasticNet)', fontsize=13, fontweight='bold')

# Graph 1: MSE
ax = axes[0]
ax.plot(t10['horizon_days'], t10['avg_cv_mse']*1000, 'b-o', ms=3, lw=1.5)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
best_mse = t10.nsmallest(1,'avg_cv_mse').iloc[0]
ax.axvline(best_mse.horizon_days, color='orange', ls=':', lw=1.5, label=f'MSE min ({int(best_mse.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('CV-MSE (x1000) — lower is better')
ax.set_title('Graph 1: Prediction Error (top10)')
ax.legend()
ax.grid(alpha=0.3)

# Graph 2: Return
ax = axes[1]
ax.plot(t10['horizon_days'], t10['avg_forward_return']*100, 'b-o', ms=3, lw=1.5)
ax.axhline(0, color='black', lw=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
best_ret = t10.nlargest(1,'avg_forward_return').iloc[0]
ax.axvline(best_ret.horizon_days, color='red', ls=':', lw=1.5, label=f'Return max ({int(best_ret.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('Avg Forward Return (%)')
ax.set_title('Graph 2: Forward Return (top10)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Graph 3 & 4: top20
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Top-20 Features  |  4 Models (OLS/Ridge/Lasso/ElasticNet)', fontsize=13, fontweight='bold')

# Graph 3: MSE
ax = axes[0]
ax.plot(t20['horizon_days'], t20['avg_cv_mse']*1000, 'r-o', ms=3, lw=1.5)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
best_mse = t20.nsmallest(1,'avg_cv_mse').iloc[0]
ax.axvline(best_mse.horizon_days, color='orange', ls=':', lw=1.5, label=f'MSE min ({int(best_mse.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('CV-MSE (x1000) — lower is better')
ax.set_title('Graph 3: Prediction Error (top20)')
ax.legend()
ax.grid(alpha=0.3)

# Graph 4: Return
ax = axes[1]
ax.plot(t20['horizon_days'], t20['avg_forward_return']*100, 'r-o', ms=3, lw=1.5)
ax.axhline(0, color='black', lw=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
best_ret = t20.nlargest(1,'avg_forward_return').iloc[0]
ax.axvline(best_ret.horizon_days, color='red', ls=':', lw=1.5, label=f'Return max ({int(best_ret.horizon_days)}d)')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('Avg Forward Return (%)')
ax.set_title('Graph 4: Forward Return (top20)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 비교: top10 vs top20 나란히
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('top10 vs top20 비교', fontsize=13, fontweight='bold')

# MSE 비교
ax = axes[0]
ax.plot(t10['horizon_days'], t10['avg_cv_mse']*1000, 'b-o', ms=3, lw=1.5, label='top10')
ax.plot(t20['horizon_days'], t20['avg_cv_mse']*1000, 'r-s', ms=3, lw=1.5, label='top20', alpha=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('CV-MSE (x1000)')
ax.set_title('MSE: top10 vs top20')
ax.legend()
ax.grid(alpha=0.3)

# Return 비교
ax = axes[1]
ax.plot(t10['horizon_days'], t10['avg_forward_return']*100, 'b-o', ms=3, lw=1.5, label='top10')
ax.plot(t20['horizon_days'], t20['avg_forward_return']*100, 'r-s', ms=3, lw=1.5, label='top20', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axvline(130, color='green', ls='--', lw=1.5, label='Current 130d')
ax.axvline(145, color='purple', ls=':', lw=1.5, label='Peak 145d')
ax.set_xlabel('Prediction Horizon (days)')
ax.set_ylabel('Avg Forward Return (%)')
ax.set_title('Return: top10 vs top20')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. 결과 분석

### 수치 요약

| 지표 | top10 | top20 |
|------|-------|-------|
| **최고 Return horizon** | **145일** (+0.93%) | **145일** (+0.86%) |
| **2위** | 150일 (+0.77%) | 150일 (+0.84%) |
| **MSE 최저 horizon** | 155일 (0.000822) | 155일 (0.000847) |
| **Dir Accuracy 최고** | 35일 (80%), 60일 (80%) | 155일 (71%) |
| **현재 시스템 130일** | 10위 (+0.46%) | 순위 밖 |
| **지배 모델** | Lasso / ElasticNet | ElasticNet |

### 패턴 해석

**Graph 1 & 3 (MSE):**
- 5~20일: MSE 낮음 — 단기 변동폭이 작아서 자연스럽게 낮음 (but 수익 없음)
- 50~130일: MSE 급등 — 예측하기 가장 어려운 구간
- **130~160일: MSE 급락** — 이 구간에서 예측 정확도 회복
- 160일 이후: MSE 다시 증가

**Graph 2 & 4 (Return):**
- 35~40일: 단기 sweet spot (방향 정확도 80%)
- 50~120일: 대체로 낮거나 음수
- **140~155일: top10/top20 모두 일관된 피크** ← 핵심 발견
- 160일 이후: 급락

### 핵심 발견

> **최적 N일 = 145일 (수익 기준) / 155일 (오차 기준)**
> 
> 현재 시스템의 130일이 나쁘지 않지만, **145~155일 구간이 더 일관되게 최적**.
> top10과 top20 모두 동일한 결론 → 견고한 발견.

---

## 5. 다음 단계

1. **즉시 가능**: 현재 시스템 `target_horizon_days=130` → `145`로 조정 시험
2. **중기**: 다른 자산 (BRK-B, QQQ, RKLB)도 같은 실험 → GLD와 최적 N일이 다른지 확인
3. **Layer 3**: SINDy로 예측 신호들을 비선형 조합 → 수익 최대화 (별도 과제)

---

## 6. 파일 경로 참조

```
research/horizon_scan_gld/
  strategy1_dense_windows.py      # Strategy1 dense 최적화 (원본 11번 코드 수정)
  run_strategy1_dense.py          # 10개 앵커 순서대로 실행
  run_horizon_scan.py             # ML horizon 스캔 마스터 (top-n 인자 지원)
  run_both_scans.py               # top10 → top20 자동 순서 실행
  outputs/
    strategy1_dense/anchor_*/     # 앵커별 dense 최적화 결과
    horizon_scan_top10_summary.csv
    horizon_scan_top20_summary.csv
    horizon_scan_graphs.png
```